In [111]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd
import os
import random
import re
from collections import Counter

SEED = 42
SAMPLES_PER_CLASS = 100
OUTPUT_DIR = "../data/SA_data"


def clean_tweet_text(text):
    """
    Limpia el texto del tweet:
    - quita saltos de línea
    - elimina TODAS las URLs del tweet_text
    - limpia espacios y comas sobrantes
    """
    text = str(text).replace("\n", " ").replace("\r", " ").strip()

    # eliminar todas las URLs
    text = re.sub(r"https?://\S+", "", text)

    # limpiar espacios antes de comas
    text = re.sub(r"\s+,", ",", text)

    # limpiar espacios duplicados
    text = re.sub(r"\s+", " ", text).strip()

    # quitar comas o espacios sobrantes al final
    text = text.rstrip(" ,")

    return text


def split_tweet_and_url(text):
    """
    Separa:
    - tweet_text: texto sin ninguna URL
    - tweet_url: última URL encontrada en el tweet
    """
    text = str(text).replace("\n", " ").replace("\r", " ").strip()

    urls = re.findall(r"https?://\S+", text)

    if urls:
        tweet_url = urls[-1].strip()
    else:
        tweet_url = ""

    tweet_text = clean_tweet_text(text)

    return pd.Series([tweet_text, tweet_url])


def load_full_damage_dataset():
    train_ds = load_dataset("QCRI/CrisisMMD", "damage", split="train")
    dev_ds = load_dataset("QCRI/CrisisMMD", "damage", split="dev")
    test_ds = load_dataset("QCRI/CrisisMMD", "damage", split="test")

    full_ds = concatenate_datasets([train_ds, dev_ds, test_ds])
    df = full_ds.to_pandas()

    if "image" in df.columns:
        df = df.drop(columns=["image"])

    df[["tweet_text", "tweet_url"]] = df["tweet_text"].apply(split_tweet_and_url)

    return df


def sample_balanced_total(df, samples_per_class=100, seed=42):
    parts = []

    for label in sorted(df["label"].unique()):
        label_df = df[df["label"] == label]

        if len(label_df) < samples_per_class:
            raise ValueError(
                f"No hay suficientes muestras para label={label}. "
                f"Disponibles: {len(label_df)}, pedidas: {samples_per_class}"
            )

        sampled = label_df.sample(n=samples_per_class, random_state=seed)
        parts.append(sampled)

    balanced_df = pd.concat(parts, ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    return balanced_df


def split_by_tweet_id(df, seed=42):
    random.seed(seed)

    groups = [g.copy() for _, g in df.groupby("tweet_id")]
    random.shuffle(groups)

    groups.sort(key=lambda g: (-len(g), -g["label"].nunique()))

    split_targets = {
        "train": {"total": 210, "labels": {0: 70, 1: 70, 2: 70}},
        "dev": {"total": 45, "labels": {0: 15, 1: 15, 2: 15}},
        "test": {"total": 45, "labels": {0: 15, 1: 15, 2: 15}},
    }

    assigned = {split: [] for split in split_targets}
    current_totals = {split: 0 for split in split_targets}
    current_labels = {split: Counter() for split in split_targets}

    def penalty(split_name, group_df):
        group_size = len(group_df)
        label_counts = group_df["label"].value_counts().to_dict()

        total_after = current_totals[split_name] + group_size
        total_target = split_targets[split_name]["total"]

        p = 0

        if total_after > total_target:
            p += (total_after - total_target) * 1000

        for label in [0, 1, 2]:
            after = current_labels[split_name][label] + label_counts.get(label, 0)
            target = split_targets[split_name]["labels"][label]

            if after > target:
                p += (after - target) * 100

            p += abs(target - after) * 0.1

        return p

    for group_df in groups:
        group_counts = group_df["label"].value_counts().to_dict()

        valid_splits = []
        for split_name in split_targets:
            fits_total = (
                current_totals[split_name] + len(group_df)
                <= split_targets[split_name]["total"]
            )
            fits_labels = all(
                current_labels[split_name][label] + group_counts.get(label, 0)
                <= split_targets[split_name]["labels"][label]
                for label in [0, 1, 2]
            )

            if fits_total and fits_labels:
                valid_splits.append(split_name)

        if valid_splits:
            chosen_split = min(valid_splits, key=lambda s: penalty(s, group_df))
        else:
            chosen_split = min(split_targets.keys(), key=lambda s: penalty(s, group_df))

        assigned[chosen_split].append(group_df)
        current_totals[chosen_split] += len(group_df)

        for label, count in group_counts.items():
            current_labels[chosen_split][label] += count

    split_dfs = {}
    for split_name in split_targets:
        if assigned[split_name]:
            split_dfs[split_name] = pd.concat(assigned[split_name], ignore_index=True)
        else:
            split_dfs[split_name] = pd.DataFrame(columns=df.columns)

    return split_dfs


def print_split_stats(split_name, df):
    print(f"\n{split_name.upper()}")
    print(f"Samples: {len(df)}")
    print("Labels:", df["label"].value_counts().sort_index().to_dict())
    print("Unique tweet_id:", df["tweet_id"].nunique())


def check_no_tweet_overlap(train_df, dev_df, test_df):
    train_ids = set(train_df["tweet_id"])
    dev_ids = set(dev_df["tweet_id"])
    test_ids = set(test_df["tweet_id"])

    print("\nOVERLAP CHECK")
    print("train-dev:", len(train_ids & dev_ids))
    print("train-test:", len(train_ids & test_ids))
    print("dev-test:", len(dev_ids & test_ids))


def save_csv_safe(df, path):
    expected_cols = [
        "event_name",
        "tweet_id",
        "image_id",
        "tweet_text",
        "tweet_url",
        "image_path",
        "label",
    ]

    df = df[expected_cols].copy()

    df["tweet_text"] = df["tweet_text"].astype(str).apply(clean_tweet_text)
    df["tweet_url"] = df["tweet_url"].astype(str).str.strip()
    df["image_path"] = df["image_path"].astype(str).str.strip()

    df.to_csv(path, index=False, encoding="utf-8")


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    full_df = load_full_damage_dataset()

    balanced_df = sample_balanced_total(
        full_df,
        samples_per_class=SAMPLES_PER_CLASS,
        seed=SEED
    )

    splits = split_by_tweet_id(balanced_df, seed=SEED)

    train_df = splits["train"]
    dev_df = splits["dev"]
    test_df = splits["test"]

    save_csv_safe(train_df, os.path.join(OUTPUT_DIR, "crisismmd_damage_train.csv"))
    save_csv_safe(dev_df, os.path.join(OUTPUT_DIR, "crisismmd_damage_dev.csv"))
    save_csv_safe(test_df, os.path.join(OUTPUT_DIR, "crisismmd_damage_test.csv"))

    print_split_stats("train", train_df)
    print_split_stats("dev", dev_df)
    print_split_stats("test", test_df)

    check_no_tweet_overlap(train_df, dev_df, test_df)

    print(f"\nArchivos guardados en: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


TRAIN
Samples: 210
Labels: {0: 70, 1: 70, 2: 70}
Unique tweet_id: 210

DEV
Samples: 45
Labels: {0: 15, 1: 15, 2: 15}
Unique tweet_id: 42

TEST
Samples: 45
Labels: {0: 15, 1: 15, 2: 15}
Unique tweet_id: 45

OVERLAP CHECK
train-dev: 0
train-test: 0
dev-test: 0

Archivos guardados en: ../data/SA_data


In [112]:
import pandas as pd
import os
import random

train_path = "../data/SA_data/crisismmd_damage_train.csv"
dev_path = "../data/SA_data/crisismmd_damage_dev.csv"
test_path = "../data/SA_data/crisismmd_damage_test.csv"

train_df = pd.read_csv(train_path)
dev_df = pd.read_csv(dev_path)
test_df = pd.read_csv(test_path)

In [113]:
print("train:", train_df.shape)
print("dev:", dev_df.shape)
print("test:", test_df.shape)

train: (210, 7)
dev: (45, 7)
test: (45, 7)


In [114]:
print(train_df.columns.tolist())

['event_name', 'tweet_id', 'image_id', 'tweet_text', 'tweet_url', 'image_path', 'label']


In [115]:
train_df.head(15)

,event_name,tweet_id,image_id,tweet_text,tweet_url,image_path,label
0,hurricane_maria,923466082358263808,923466082358263808_0,Questions over $300m Puerto Rico contract handed to two-person firm #Scotland,https://t.co/82eHTxCmuq,data_image/hurricane_maria/26_10_2017/923466082358263808_0.jpg,0
1,hurricane_harvey,905930890735439873,905930890735439873_2,Some of the devastation from #Harvey friends and family who lost everything ὢ2 prayers neededὉ4 @DonnieWahlberg #BH❤,https://t.co/2rNeqYEEQJ,data_image/hurricane_harvey/7_9_2017/905930890735439873_2.jpg,0
2,hurricane_maria,911711789246693377,911711789246693377_0,Dominican Republic: 38 towns isolated after #HurricaneMaria,https://t.co/qhzfmmo52R,data_image/hurricane_maria/23_9_2017/911711789246693377_0.jpg,0
3,hurricane_harvey,905556047984824322,905556047984824322_1,"RT @MedinaStormCast: @weatherchannel Crestline, OH tornado damage.",https://t.co/6anRQAf3te,data_image/hurricane_harvey/6_9_2017/905556047984824322_1.jpg,0
4,hurricane_maria,915319640145911809,915319640145911809_1,RT @newxseason: Trump in Puerto Rico: “this isn’t a REAL catastrophe like Katrina” ὄ7ἿCὄ7ἿCὄ7ἿCὄ7ἿC What? ᾓ7ἿD‍♀️,https://t.co/KzCwt6jHKb,data_image/hurricane_maria/3_10_2017/915319640145911809_1.jpg,1
5,hurricane_harvey,908270102289805312,908270102289805312_0,Property/casualty group Travelers sees up to $750m in losses from Hurricane Harvey,https://t.co/5xCY76mJat,data_image/hurricane_harvey/14_9_2017/908270102289805312_0.jpg,2
6,mexico_earthquake,912048104651788289,912048104651788289_2,Extra images of the leaning building #MexicoEarthquake,https://t.co/SibrGMDgjv,data_image/mexico_earthquake/24_9_2017/912048104651788289_2.jpg,0
7,hurricane_irma,909880224631857152,909880224631857152_0,"Restoration of Water Supply Benefits 25,000 Cubans Affected by Irma",https://t.co/SdqjlogwpZ,data_image/hurricane_irma/18_9_2017/909880224631857152_0.jpg,0
8,hurricane_maria,910582078470852608,910582078470852608_0,Seven Die and Thousands Hurricane-Stricken in Dominica After Maria -,https://t.co/YtuDznHy2X,data_image/hurricane_maria/20_9_2017/910582078470852608_0.jpg,1
9,hurricane_harvey,906934889919991808,906934889919991808_0,Analysts: Hurricane Harvey could slow economic growth by full percentage point,https://t.co/xFaettK2sp,data_image/hurricane_harvey/10_9_2017/906934889919991808_0.jpg,0


In [116]:
print("train")
print(train_df["label"].value_counts().sort_index())

print("\ndev")
print(dev_df["label"].value_counts().sort_index())

print("\ntest")
print(test_df["label"].value_counts().sort_index())

train
label
0    70
1    70
2    70
Name: count, dtype: int64

dev
label
0    15
1    15
2    15
Name: count, dtype: int64

test
label
0    15
1    15
2    15
Name: count, dtype: int64


In [117]:
train_ids = set(train_df["tweet_id"])
dev_ids = set(dev_df["tweet_id"])
test_ids = set(test_df["tweet_id"])

print("train-dev:", len(train_ids & dev_ids))
print("train-test:", len(train_ids & test_ids))
print("dev-test:", len(dev_ids & test_ids))

train-dev: 0
train-test: 0
dev-test: 0


In [118]:
print("train duplicated tweet_id:", train_df["tweet_id"].duplicated().sum())
print("dev duplicated tweet_id:", dev_df["tweet_id"].duplicated().sum())
print("test duplicated tweet_id:", test_df["tweet_id"].duplicated().sum())

train duplicated tweet_id: 0
dev duplicated tweet_id: 3
test duplicated tweet_id: 0


In [119]:
dev_df[dev_df["tweet_id"].duplicated(keep=False)].sort_values("tweet_id")

,event_name,tweet_id,image_id,tweet_text,tweet_url,image_path,label
4,hurricane_harvey,901688029521334272,901688029521334272_2,RT @slizzards96: ummmm so a tornado just went through my backyard like bitch all i was expecting was a hurricane,https://t.co/6rqYiYVXbG,data_image/hurricane_harvey/27_8_2017/901688029521334272_2.jpg,0
5,hurricane_harvey,901688029521334272,901688029521334272_3,RT @slizzards96: ummmm so a tornado just went through my backyard like bitch all i was expecting was a hurricane,https://t.co/6rqYiYVXbG,data_image/hurricane_harvey/27_8_2017/901688029521334272_3.jpg,0
0,hurricane_irma,909968982362517504,909968982362517504_3,A glimpse of the damage #Irma did to our beautiful island...,https://t.co/6lHx9zXjDw,data_image/hurricane_irma/19_9_2017/909968982362517504_3.jpg,1
1,hurricane_irma,909968982362517504,909968982362517504_0,A glimpse of the damage #Irma did to our beautiful island...,https://t.co/6lHx9zXjDw,data_image/hurricane_irma/19_9_2017/909968982362517504_0.jpg,0
2,hurricane_maria,913549726083141632,913549726083141632_2,Everything was so green and full of life before #HurricaneMaria. Now it’s been stripped away. ὡ4,https://t.co/5nsiNwuDp1,data_image/hurricane_maria/28_9_2017/913549726083141632_2.jpg,0
3,hurricane_maria,913549726083141632,913549726083141632_1,Everything was so green and full of life before #HurricaneMaria. Now it’s been stripped away. ὡ4,https://t.co/5nsiNwuDp1,data_image/hurricane_maria/28_9_2017/913549726083141632_1.jpg,1


In [120]:
print("train")
print(train_df.isnull().sum())

print("\ndev")
print(dev_df.isnull().sum())

print("\ntest")
print(test_df.isnull().sum())

train
event_name    0
tweet_id      0
image_id      0
tweet_text    0
tweet_url     0
image_path    0
label         0
dtype: int64

dev
event_name    0
tweet_id      0
image_id      0
tweet_text    0
tweet_url     0
image_path    0
label         0
dtype: int64

test
event_name    0
tweet_id      0
image_id      0
tweet_text    0
tweet_url     0
image_path    0
label         0
dtype: int64


In [121]:
print("Empty train texts:", (train_df["tweet_text"].astype(str).str.strip() == "").sum())
print("Empty dev texts:", (dev_df["tweet_text"].astype(str).str.strip() == "").sum())
print("Empty test texts:", (test_df["tweet_text"].astype(str).str.strip() == "").sum())

Empty train texts: 0
Empty dev texts: 0
Empty test texts: 0


In [122]:
print("train missing images:", (~train_df["image_path"].apply(os.path.exists)).sum())
print("dev missing images:", (~dev_df["image_path"].apply(os.path.exists)).sum())
print("test missing images:", (~test_df["image_path"].apply(os.path.exists)).sum())

train missing images: 210
dev missing images: 45
test missing images: 45


In [123]:
print("train")
print(train_df["event_name"].value_counts())

print("\ndev")
print(dev_df["event_name"].value_counts())

print("\ntest")
print(test_df["event_name"].value_counts())

train
event_name
hurricane_irma          67
hurricane_harvey        62
hurricane_maria         53
california_wildfires    18
mexico_earthquake        5
srilanka_floods          3
iraq_iran_earthquake     2
Name: count, dtype: int64

dev
event_name
hurricane_irma          14
hurricane_harvey        12
hurricane_maria         11
california_wildfires     3
mexico_earthquake        3
iraq_iran_earthquake     1
srilanka_floods          1
Name: count, dtype: int64

test
event_name
hurricane_irma          14
hurricane_harvey        12
hurricane_maria          8
california_wildfires     5
iraq_iran_earthquake     3
mexico_earthquake        2
srilanka_floods          1
Name: count, dtype: int64


In [124]:
for df in [train_df, dev_df, test_df]:
    df["tweet_text_clean"] = (
        df["tweet_text"]
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [125]:
pd.set_option("display.max_colwidth", None)
train_df[["tweet_text", "tweet_text_clean"]].head(5)

,tweet_text,tweet_text_clean
0,Questions over $300m Puerto Rico contract handed to two-person firm #Scotland,Questions over $300m Puerto Rico contract handed to two-person firm #Scotland
1,Some of the devastation from #Harvey friends and family who lost everything ὢ2 prayers neededὉ4 @DonnieWahlberg #BH❤,Some of the devastation from #Harvey friends and family who lost everything ὢ2 prayers neededὉ4 @DonnieWahlberg #BH❤
2,Dominican Republic: 38 towns isolated after #HurricaneMaria,Dominican Republic: 38 towns isolated after #HurricaneMaria
3,"RT @MedinaStormCast: @weatherchannel Crestline, OH tornado damage.","RT @MedinaStormCast: @weatherchannel Crestline, OH tornado damage."
4,RT @newxseason: Trump in Puerto Rico: “this isn’t a REAL catastrophe like Katrina” ὄ7ἿCὄ7ἿCὄ7ἿCὄ7ἿC What? ᾓ7ἿD‍♀️,RT @newxseason: Trump in Puerto Rico: “this isn’t a REAL catastrophe like Katrina” ὄ7ἿCὄ7ἿCὄ7ἿCὄ7ἿC What? ᾓ7ἿD‍♀️


In [126]:
train_df[["tweet_text", "tweet_text_clean"]].head(5)

,tweet_text,tweet_text_clean
0,Questions over $300m Puerto Rico contract handed to two-person firm #Scotland,Questions over $300m Puerto Rico contract handed to two-person firm #Scotland
1,Some of the devastation from #Harvey friends and family who lost everything ὢ2 prayers neededὉ4 @DonnieWahlberg #BH❤,Some of the devastation from #Harvey friends and family who lost everything ὢ2 prayers neededὉ4 @DonnieWahlberg #BH❤
2,Dominican Republic: 38 towns isolated after #HurricaneMaria,Dominican Republic: 38 towns isolated after #HurricaneMaria
3,"RT @MedinaStormCast: @weatherchannel Crestline, OH tornado damage.","RT @MedinaStormCast: @weatherchannel Crestline, OH tornado damage."
4,RT @newxseason: Trump in Puerto Rico: “this isn’t a REAL catastrophe like Katrina” ὄ7ἿCὄ7ἿCὄ7ἿCὄ7ἿC What? ᾓ7ἿD‍♀️,RT @newxseason: Trump in Puerto Rico: “this isn’t a REAL catastrophe like Katrina” ὄ7ἿCὄ7ἿCὄ7ἿCὄ7ἿC What? ᾓ7ἿD‍♀️


In [127]:
(train_df["tweet_text"] != train_df["tweet_text_clean"]).sum()

0

In [128]:
import re

def clean_tweet_text(text):
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"http\S+|www\.\S+", " <URL> ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for df in [train_df, dev_df, test_df]:
    df["tweet_text_clean_v2"] = df["tweet_text"].apply(clean_tweet_text)

In [129]:
train_df[["tweet_text", "tweet_text_clean_v2"]].head(20)

,tweet_text,tweet_text_clean_v2
0,Questions over $300m Puerto Rico contract handed to two-person firm #Scotland,Questions over $300m Puerto Rico contract handed to two-person firm #Scotland
1,Some of the devastation from #Harvey friends and family who lost everything ὢ2 prayers neededὉ4 @DonnieWahlberg #BH❤,Some of the devastation from #Harvey friends and family who lost everything ὢ2 prayers neededὉ4 @DonnieWahlberg #BH❤
2,Dominican Republic: 38 towns isolated after #HurricaneMaria,Dominican Republic: 38 towns isolated after #HurricaneMaria
3,"RT @MedinaStormCast: @weatherchannel Crestline, OH tornado damage.","RT @MedinaStormCast: @weatherchannel Crestline, OH tornado damage."
4,RT @newxseason: Trump in Puerto Rico: “this isn’t a REAL catastrophe like Katrina” ὄ7ἿCὄ7ἿCὄ7ἿCὄ7ἿC What? ᾓ7ἿD‍♀️,RT @newxseason: Trump in Puerto Rico: “this isn’t a REAL catastrophe like Katrina” ὄ7ἿCὄ7ἿCὄ7ἿCὄ7ἿC What? ᾓ7ἿD‍♀️
5,Property/casualty group Travelers sees up to $750m in losses from Hurricane Harvey,Property/casualty group Travelers sees up to $750m in losses from Hurricane Harvey
6,Extra images of the leaning building #MexicoEarthquake,Extra images of the leaning building #MexicoEarthquake
7,"Restoration of Water Supply Benefits 25,000 Cubans Affected by Irma","Restoration of Water Supply Benefits 25,000 Cubans Affected by Irma"
8,Seven Die and Thousands Hurricane-Stricken in Dominica After Maria -,Seven Die and Thousands Hurricane-Stricken in Dominica After Maria -
9,Analysts: Hurricane Harvey could slow economic growth by full percentage point,Analysts: Hurricane Harvey could slow economic growth by full percentage point


In [130]:
weird_pattern = r"[ðÐ�ὄ]"
(train_df["tweet_text"].astype(str).str.contains(weird_pattern, regex=True)).sum()

2

In [131]:
print("train missing images:", (~train_df["image_path"].apply(os.path.exists)).sum())
print("dev missing images:", (~dev_df["image_path"].apply(os.path.exists)).sum())
print("test missing images:", (~test_df["image_path"].apply(os.path.exists)).sum())

train missing images: 210
dev missing images: 45
test missing images: 45


In [132]:
train_df["image_path"].head(10).tolist()

['data_image/hurricane_maria/26_10_2017/923466082358263808_0.jpg',
 'data_image/hurricane_harvey/7_9_2017/905930890735439873_2.jpg',
 'data_image/hurricane_maria/23_9_2017/911711789246693377_0.jpg',
 'data_image/hurricane_harvey/6_9_2017/905556047984824322_1.jpg',
 'data_image/hurricane_maria/3_10_2017/915319640145911809_1.jpg',
 'data_image/hurricane_harvey/14_9_2017/908270102289805312_0.jpg',
 'data_image/mexico_earthquake/24_9_2017/912048104651788289_2.jpg',
 'data_image/hurricane_irma/18_9_2017/909880224631857152_0.jpg',
 'data_image/hurricane_maria/20_9_2017/910582078470852608_0.jpg',
 'data_image/hurricane_harvey/10_9_2017/906934889919991808_0.jpg']

In [133]:
import pandas as pd
import os

train_df = pd.read_csv("../data/SA_data/crisismmd_damage_train.csv")
dev_df = pd.read_csv("../data/SA_data/crisismmd_damage_dev.csv")
test_df = pd.read_csv("../data/SA_data/crisismmd_damage_test.csv")

print("train missing:", (~train_df["image_path"].fillna("").apply(os.path.exists)).sum())
print("dev missing:", (~dev_df["image_path"].fillna("").apply(os.path.exists)).sum())
print("test missing:", (~test_df["image_path"].fillna("").apply(os.path.exists)).sum())

train missing: 210
dev missing: 45
test missing: 45


In [134]:
from datasets import load_dataset

ds = load_dataset("QCRI/CrisisMMD", "damage", split="train")
print(ds.column_names)
print(ds.features)
print(ds[0])

['event_name', 'tweet_id', 'image_id', 'tweet_text', 'image_path', 'image', 'label']
{'event_name': Value('string'), 'tweet_id': Value('string'), 'image_id': Value('string'), 'tweet_text': Value('string'), 'image_path': Value('string'), 'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['little_or_no_damage', 'mild_damage', 'severe_damage'])}
{'event_name': 'hurricane_harvey', 'tweet_id': '905960092822003712', 'image_id': '905960092822003712_0', 'tweet_text': 'RT @tveitdal: Where Harvey is hitting hardest, 80 percent lack flood insurance https://t.co/p0xVVe3kNP https://t.co/xaHjtRRGMq', 'image_path': 'data_image/hurricane_harvey/8_9_2017/905960092822003712_0.jpg', 'image': None, 'label': 2}
